In [ ]:
import pandas as pd
import refinitiv.data as rd

universe_of_indices = [
    '0#.SPSUP',   # US (All)
    '0#.STOXX',   # Europe (All) 
    '0#.GSPTSE',  # Canada (Large)
    '0#.SPTSEM',  # Canada (Mid)
    '0#.SPTSES',  # Canada (Small)
    '0#.N225',    # Japan (Nikkei 225 - Large)
    '0#.TOPX1000' # Japan (TOPIX 1000 - Small to Large)
]

if len(universe_of_indices) > 0:

    rd.open_session()
    
    df = rd.get_data(
        universe = universe_of_indices,
        
        fields = [
                'TR.IndexConstituentRIC', # (Index Constituents RICs)
                'TR.IndexConstituentName' # (Index Constituents Names)
                #'TR.FundHoldingRIC', 
                #'TR.FundHoldingName'
            ]
    )

    rics_list = df['Instrument'].tolist()

    # Get data from Refinitiv
    refinitiv_universe  = rd.get_data(
        universe=rics_list, 
        fields=[
            'TR.ISINcode',
            'TR.CommonName'
        ]
    )
    print(refinitiv_universe)
    print(refinitiv_universe.shape[0])

else:
    refinitiv_universe = pd.DataFrame()
    print('No indices selected')

In [ ]:
# Load isins from csv under cr/input
ishares_universe = pd.concat(
    [
        pd.read_csv('cr/input/ishares_core_sp_asx_200.csv'),  # Australia
        pd.read_csv('cr/input/ishares_msci_new_zealand.csv'), # New Zealand
        pd.read_csv('cr/input/ishares_msci_korea.csv'),       # South Korea
    ],
    axis=0,
    ignore_index=True
)

print(ishares_universe)
print(ishares_universe.shape[0])

In [ ]:
# Append Refinitiv and iShares universes
merged_universe = pd.concat([refinitiv_universe, ishares_universe], axis=0, ignore_index=True)
print(merged_universe)
print(merged_universe.shape[0])

In [ ]:
# Load LC dataset
lc = pd.read_csv('./data/LC_dataset_v_1_1O.csv')

# Current list of isins
current_isin_list = []

# Iterate through the 'full_isin_list' Series directly
for item in lc['full_isin_list'].dropna(): # dropna() removes NaN values
    if isinstance(item, str) and item.strip(): # Check if it's a non-empty string
        current_isin_list.extend([isin.strip() for isin in item.split(',')])

# Keep the isins in `merged_universe` that are not in `current_isin_list`
isin_list = [isin for isin in merged_universe['ISIN Code'].tolist() if isin not in current_isin_list]
isin_list

In [ ]:
current_isin_list

In [ ]:
# Take `merged_universe` rows such that the isin is in `isin_list`
isin_df = merged_universe[merged_universe['ISIN Code'].isin(isin_list)].reset_index(drop=True)
isin_df.to_csv('./cr/new_isins.csv', index=False)
print(isin_df.shape[0])

print('Missing ISINs:')
print(f'{100*isin_df.shape[0]/merged_universe.shape[0]}%')

In [ ]:
'''
--------------------------------

Missing ISINs for Australia:
43.48%

Missing ISINs for Canada:
53.03%

Missing ISINs for Japan:
48.29%

Missing ISINs for New Zealand:
36.00%

Missing ISINs for South Korea:
42.68%

--------------------------------

Total missing ISINs:
48.30%

--------------------------------
''';

In [ ]:
# Cleanup
lc['full_isin_list'] = lc['full_isin_list'].astype(str)
lc = lc.dropna(subset=['rfyear', 'searched_company_name', 'full_isin_list'], how='any')
lc = lc[lc['full_isin_list'] != 'nan']
lc = lc.drop_duplicates(subset=['rfyear', 'full_isin_list'])

# Get all unique values for rfyear and searched_company_name
rfyear_vals = lc['rfyear'].unique()
isin_vals = lc['full_isin_list'].unique()

# Create a MultiIndex from the Cartesian product
multi_index = pd.MultiIndex.from_product([rfyear_vals, isin_vals],
                                           names=['rfyear', 'full_isin_list'])

# Set the DataFrame index
lc = lc.set_index(['rfyear', 'full_isin_list'])

# Reindex to include all combinations, missing values will be NaN
lc = lc.reindex(multi_index).reset_index()

In [ ]:
def fill_with_first(s):
    non_nan = s.dropna()
    if not non_nan.empty:
        return s.fillna(non_nan.iloc[0])
    else:
        return s

lc['searched_company_name'] = lc.groupby('full_isin_list')['searched_company_name'].transform(fill_with_first)

In [ ]:
# List of country codes to filter by
country_codes = sorted(isin_df['ISIN Code'].str[:2].unique())

# Filter rows where 'TYPE: r&d investments' is NaN
lc_missing_coverage = lc[lc['TYPE: r&d investments'].isna()]

# Create a single DataFrame containing rows where any of the country codes is in 'full_isin_list'
lc_missing_coverage = lc_missing_coverage[
    lc_missing_coverage['full_isin_list'].apply(lambda x: any(code in x for code in country_codes))
]

lc_missing_coverage.loc[
    (lc_missing_coverage['rfyear'] >= 2009), 
    ['rfyear', 'full_isin_list', 'searched_company_name']
].sort_values(by=['rfyear', 'full_isin_list']).to_csv('./cr/incomplete_coverage.csv', index=False)